In [1]:
from fixture.factory.dataset.ohlcv import factory_ohlcv_cycle

ohlcv = factory_ohlcv_cycle()
ohlcv.df

Date,Open,High,Low,Close,Volume
datetime[μs],f64,f64,f64,f64,i64
2000-01-01 00:00:00,101.0,101.19,99.4,99.68,10552
2000-01-02 00:00:00,99.68,100.86,99.45,100.84,21735
2000-01-03 00:00:00,100.84,101.15,100.09,100.21,15913
2000-01-04 00:00:00,99.21,100.91,99.17,100.76,42179
2000-01-05 00:00:00,100.76,101.24,100.73,101.16,26119
…,…,…,…,…,…
2000-04-05 00:00:00,109.1,109.17,107.75,107.9,40443
2000-04-06 00:00:00,107.9,109.27,107.48,109.17,48873
2000-04-07 00:00:00,109.17,109.42,108.71,108.71,25101


In [2]:
from domain.feature.closes.service import derive_closes


df_closes = derive_closes(ohlcv, 8)
df_closes

Date,now,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8
datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64
2000-01-10 00:00:00,192.798497,-101.311951,0.988289,-15.800912,17.777782,39.619703,54.734674,-62.671182,115.700473
2000-01-11 00:00:00,-62.868576,192.798497,-101.311951,0.988289,-15.800912,17.777782,39.619703,54.734674,-62.671182
2000-01-12 00:00:00,23.622058,-62.868576,192.798497,-101.311951,0.988289,-15.800912,17.777782,39.619703,54.734674
2000-01-13 00:00:00,-4.916663,23.622058,-62.868576,192.798497,-101.311951,0.988289,-15.800912,17.777782,39.619703
2000-01-14 00:00:00,16.7068,-4.916663,23.622058,-62.868576,192.798497,-101.311951,0.988289,-15.800912,17.777782
…,…,…,…,…,…,…,…,…,…
2000-04-05 00:00:00,-11.115229,138.897595,9.391436,46.145959,0.943975,122.526109,-46.717919,125.390492,43.438458
2000-04-06 00:00:00,117.01428,-11.115229,138.897595,9.391436,46.145959,0.943975,122.526109,-46.717919,125.390492
2000-04-07 00:00:00,-42.225141,117.01428,-11.115229,138.897595,9.391436,46.145959,0.943975,122.526109,-46.717919


In [3]:
from domain.feature.closes.derive import derive_closes_n4

closes_n4 = derive_closes_n4(ohlcv)
closes_n4.df

Date,now,lag_1,lag_2,lag_3,lag_4
datetime[μs],f64,f64,f64,f64,f64
2000-01-06 00:00:00,17.777782,39.619703,54.734674,-62.671182,115.700473
2000-01-07 00:00:00,-15.800912,17.777782,39.619703,54.734674,-62.671182
2000-01-08 00:00:00,0.988289,-15.800912,17.777782,39.619703,54.734674
2000-01-09 00:00:00,-101.311951,0.988289,-15.800912,17.777782,39.619703
2000-01-10 00:00:00,192.798497,-101.311951,0.988289,-15.800912,17.777782
…,…,…,…,…,…
2000-04-05 00:00:00,-11.115229,138.897595,9.391436,46.145959,0.943975
2000-04-06 00:00:00,117.01428,-11.115229,138.897595,9.391436,46.145959
2000-04-07 00:00:00,-42.225141,117.01428,-11.115229,138.897595,9.391436


In [4]:
df_closes.equals(closes_n4.df)

False

In [5]:
# closesの特徴量分析

import plotly.express as px
import plotly.graph_objects as go
import polars as pl

# 相関行列ヒートマップ
corr = df_closes.drop("Date").to_pandas().corr()
fig = px.imshow(
    corr,
    labels=dict(x="Features", y="Features", color="Correlation"),
    x=corr.columns,
    y=corr.columns,
    title="特徴量間の相関関係ヒートマップ",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
)
fig.update_layout(width=800, height=800)
fig.show()

# 特徴量分布のボックスプロット
fig = go.Figure()
for col in df_closes.columns[1:]:  # Dateを除外
    fig.add_trace(
        go.Box(
            y=df_closes[col], name=col, boxpoints="outliers", jitter=0.3, pointpos=-1.8
        )
    )
fig.update_layout(title="特徴量の分布と外れ値", yaxis_title="Value", boxmode="group")
fig.show()

# 時系列プロット（now特徴量）
fig = px.line(
    df_closes.to_pandas(),
    x="Date",
    y="now",
    title="'now'特徴量の時系列変化",
    labels={"now": "Value"},
    template="plotly_white",
)
# 移動平均を計算してプロット
fig.add_trace(
    go.Scatter(
        x=df_closes["Date"],
        y=df_closes["now"].rolling_mean(5),
        mode="lines",
        name="5日移動平均",
        line=dict(color="red", dash="dot"),
    )
)
fig.update_layout(xaxis_title="Date", yaxis_title="Value", hovermode="x unified")
fig.show()

# ラグ特徴量の相互作用（3D散布図）
fig = px.scatter_3d(
    df_closes.head(100).to_pandas(),
    x="lag_1",
    y="lag_2",
    z="now",
    color="lag_3",
    title="ラグ特徴量の3D相互作用",
    labels={"lag_1": "Lag 1", "lag_2": "Lag 2", "now": "Current"},
    color_continuous_scale=px.colors.sequential.Viridis,
)
fig.update_layout(
    scene=dict(xaxis_title="Lag 1", yaxis_title="Lag 2", zaxis_title="Current Value"),
    width=1000,
    height=800,
)
fig.show()

In [6]:
# 基本統計量の確認（修正版）
stats = df_closes.select(
    [
        pl.all().exclude("Date").mean().name.suffix("_mean"),
        pl.all().exclude("Date").std().name.suffix("_std"),
        pl.all().exclude("Date").skew().name.suffix("_skew"),
        pl.all().exclude("Date").kurtosis().name.suffix("_kurt"),
    ]
).transpose(include_header=True, header_name="feature")
print("基本統計量:")
print(stats)

autocorr = df_closes.select(
    [
        pl.corr(pl.col("now"), pl.col("now").shift(1)).alias("lag1"),
        pl.corr(pl.col("now"), pl.col("now").shift(2)).alias("lag2"),
        pl.corr(pl.col("now"), pl.col("now").shift(3)).alias("lag3"),
        pl.corr(pl.col("now"), pl.col("now").shift(4)).alias("lag4"),
    ]
)
print("\n自己相関係数:")
print(autocorr)

# 分散比（前日との変動幅比較）
variance_ratio = df_closes.select(
    [
        (pl.col("now").var() / pl.col("lag_1").var()).alias("var_ratio_now_vs_lag1"),
        (pl.col("lag_1").var() / pl.col("lag_2").var()).alias("var_ratio_lag1_vs_lag2"),
    ]
)
print("\n分散比:")
print(variance_ratio)

# 異常値カウント（3σ以上）
outliers = df_closes.select(
    [
        pl.col("now")
        .filter(pl.col("now").abs() > 3 * pl.col("now").std())
        .count()
        .alias("now_outliers"),
        pl.col("lag_1")
        .filter(pl.col("lag_1").abs() > 3 * pl.col("lag_1").std())
        .count()
        .alias("lag1_outliers"),
    ]
)
print("\n異常値数（3σ以上）:")
print(outliers)

# トレンド継続日数分析
# trend_days = (
#     closes
#     .with_columns(
#         (pl.col("now") > 0).cast(pl.UInt8).alias("up_flag")
#     )
#     .with_columns(
#         pl.cumsum(
#             (pl.col("up_flag") != pl.col("up_flag").shift(1)).cast(pl.UInt8)
#         ).alias("run_id")
#     )
#     .groupby("run_id", "up_flag")
#     .agg(
#         consecutive_days = pl.count(),
#     )
#     .filter(pl.col("up_flag") == 1)
#     .select("consecutive_days")
# )
# print("\n上昇継続日数分布:")
# print(trend_days)

基本統計量:
shape: (36, 2)
┌────────────┬───────────┐
│ feature    ┆ column_0  │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ now_mean   ┆ 10.187046 │
│ lag_1_mean ┆ 8.291017  │
│ lag_2_mean ┆ 7.888206  │
│ lag_3_mean ┆ 8.178582  │
│ lag_4_mean ┆ 7.088071  │
│ …          ┆ …         │
│ lag_4_kurt ┆ -0.412433 │
│ lag_5_kurt ┆ -0.43017  │
│ lag_6_kurt ┆ -0.359352 │
│ lag_7_kurt ┆ -0.383326 │
│ lag_8_kurt ┆ -0.454418 │
└────────────┴───────────┘

自己相関係数:
shape: (1, 4)
┌───────────┬──────────┬──────────┬──────────┐
│ lag1      ┆ lag2     ┆ lag3     ┆ lag4     │
│ ---       ┆ ---      ┆ ---      ┆ ---      │
│ f64       ┆ f64      ┆ f64      ┆ f64      │
╞═══════════╪══════════╪══════════╪══════════╡
│ -0.246424 ┆ 0.188345 ┆ 0.139293 ┆ 0.138193 │
└───────────┴──────────┴──────────┴──────────┘

分散比:
shape: (1, 2)
┌───────────────────────┬────────────────────────┐
│ var_ratio_now_vs_lag1 ┆ var_ratio_lag1_vs_lag2 │
│ ---                   ┆ ---                